In [2]:
import pandas as pd
import numpy as np
import re
from datetime import datetime


In [3]:
# -------- CONFIG --------
EMAILS_FILE = "emails_test.csv"
REPLIES_FILE = "replies_test.csv"
OUTPUT_FILE = "cleaned_pairs.csv"

In [4]:
def clean_text(text):
    if pd.isna(text): return ""
    s = str(text)

    # remove quoted blocks (reply chains)
    s = re.sub(r'On .* wrote:.*', '', s, flags=re.DOTALL)
    s = re.sub(r'>+.*', '', s)

    # remove email signature patterns
    s = re.sub(r'(--|__|Thanks,|Best,|Regards,|Sent from my).*', '', s, flags=re.IGNORECASE)

    # normalize whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def parse_date_safe(x):
    try:
        return pd.to_datetime(x, errors='coerce', utc=True)
    except Exception:
        return pd.NaT


In [5]:
# 1. Load data
emails = pd.read_csv(EMAILS_FILE)
replies = pd.read_csv(REPLIES_FILE)

# 2. Clean column names (remove stray spaces)
emails.columns = emails.columns.str.strip()
replies.columns = replies.columns.str.strip()

# 3. Merge datasets
merged = replies.merge(emails, left_on='Email_ID', right_on='ID', how='left', suffixes=('_reply', '_email'))

# 4. Drop if no matching email
merged = merged.dropna(subset=['Body', 'Reply_Body'])

# 5. Clean text
merged['email_subject_clean'] = merged['Subject'].apply(clean_text)
merged['email_body_clean'] = merged['Body'].apply(clean_text)
merged['reply_subject_clean'] = merged['Reply_Subject'].apply(clean_text)
merged['reply_body_clean'] = merged['Reply_Body'].apply(clean_text)

# 6. Convert dates
merged['Date_email'] = merged['Date_email'].apply(parse_date_safe)
merged['Date_reply'] = merged['Date_reply'].apply(parse_date_safe)

# 7. Compute response delay (in hours)
merged['response_delay_hrs'] = (merged['Date_reply'] - merged['Date_email']).dt.total_seconds() / 3600.0

# 8. Drop irrelevant or empty entries
merged = merged.dropna(subset=['email_body_clean', 'reply_body_clean'])
merged = merged[merged['email_body_clean'].str.len() > 10]
merged = merged[merged['reply_body_clean'].str.len() > 10]

# 9. Add pair_id
merged = merged.reset_index(drop=True)
merged['pair_id'] = merged.index.astype(int)

# 10. Select and rename columns
final = merged[[
    'pair_id',
    'email_subject_clean', 'email_body_clean', 'Sender', 'Date_email',
    'reply_subject_clean', 'reply_body_clean', 'Responder', 'Date_reply',
    'response_delay_hrs'
]].rename(columns={
    'Sender': 'email_sender',
    'Responder': 'reply_sender'
})

# 11. Save cleaned output
final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"✅ Cleaned dataset saved to {OUTPUT_FILE}")
print(f"Total pairs after cleaning: {len(final)}")

# Optional: quick summary
print(final.head(3))

✅ Cleaned dataset saved to cleaned_pairs.csv
Total pairs after cleaning: 86376
   pair_id             email_subject_clean  \
0        0  Schedule sync for release plan   
1        1  Schedule sync for release plan   
2        2  Schedule sync for release plan   

                                    email_body_clean        email_sender  \
0  Hello, Requesting a sync about release plan. P...  anita.jain@acme.co   
1  Hello, Requesting a sync about release plan. P...  anita.jain@acme.co   
2  Hello, Requesting a sync about release plan. P...  anita.jain@acme.co   

                 Date_email                        reply_subject_clean  \
0 2025-07-18 00:00:00+00:00           Confirming: release plan meeting   
1 2025-07-18 00:00:00+00:00  Can't make 2025-06-12 - propose alternate   
2 2025-07-18 00:00:00+00:00  Can't make 2025-08-03 - propose alternate   

                                    reply_body_clean       reply_sender  \
0  Hi Anita Jain, I can take the meeting on 11:00...       